# 0. Libraries

## 0.1. Installing

In [1]:
!pip install agno==1.7.7 google-genai requests pypdf chromadb pandas -q

## 0.2. Importing

In [2]:
import json

import pandas as pd
import numpy as np

from datetime import datetime

from typing import Literal

from agno.agent import Agent
from agno.models.google import Gemini
from agno.models.openai import OpenAIChat

from agno.team.team import Team

from agno.tools.reasoning import ReasoningTools

from pydantic import BaseModel, Field

In [49]:
import os

os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

In [4]:
data = pd.read_csv("lead1.0-small.csv")

In [5]:
data

,building_id,timestamp,meter_reading,anomaly
0,1,2016-01-01 00:00:00,NaN,0
1,32,2016-01-01 00:00:00,NaN,0
2,41,2016-01-01 00:00:00,NaN,0
3,55,2016-01-01 00:00:00,NaN,0
4,69,2016-01-01 00:00:00,NaN,0
...,...,...,...,...
1749489,1316,2016-12-31 23:00:00,38.844,0
1749490,1318,2016-12-31 23:00:00,202.893,0
1749491,1319,2016-12-31 23:00:00,NaN,0
1749492,1323,2016-12-31 23:00:00,172.000,0


In [50]:
data["timestamp"] = pd.to_datetime(data["timestamp"], format="%Y-%m-%d %H:%M:%S")

/tmp/ipython-input-2323623616.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["timestamp"] = pd.to_datetime(data["timestamp"], format="%Y-%m-%d %H:%M:%S")


In [51]:
data = data[~data["meter_reading"].isna()]

In [52]:
data

,building_id,timestamp,meter_reading,anomaly
8,107,2016-01-01 00:00:00,175.184,1
10,111,2016-01-01 00:00:00,167.392,1
11,112,2016-01-01 00:00:00,10.275,0
12,117,2016-01-01 00:00:00,16.306,0
13,118,2016-01-01 00:00:00,117.200,0
...,...,...,...,...
1749488,1315,2016-12-31 23:00:00,32.520,0
1749489,1316,2016-12-31 23:00:00,38.844,0
1749490,1318,2016-12-31 23:00:00,202.893,0
1749492,1323,2016-12-31 23:00:00,172.000,0


In [53]:
def filter_data_by_timestamp(
  timestamp,
  building_id: int = None,
  range_days: int = 7,
  only_anomaly: bool = False
) -> pd.DataFrame:
  start_range = pd.to_datetime(timestamp)

  end_range = start_range.normalize() + pd.Timedelta(days=range_days)

  filtered_df = data.copy()

  if building_id:
      filtered_df = filtered_df[filtered_df["building_id"] == building_id]

  if only_anomaly:
      filtered_df = filtered_df[filtered_df["anomaly"] == only_anomaly]

  filtered_df = filtered_df[
      (filtered_df["timestamp"] >= start_range) & (filtered_df["timestamp"] < end_range)
  ]

  return filtered_df

In [54]:
def get_meter_reading_statistics(
  timestamp: str,
  building_id: int | None = None,
  range_days: int = 7,
  only_anomaly: bool = False
) -> dict:
  filtered_df = filter_data_by_timestamp(
      timestamp=timestamp,
      building_id=building_id,
      range_days=range_days,
      only_anomaly=only_anomaly
  )

  data_result = {}

  if building_id:
    data_result["building_id"] = building_id

  if range_days:
    data_result["range_days"] = range_days

  data_result["maximum"] = filtered_df["meter_reading"].max()
  data_result["average"] = filtered_df["meter_reading"].mean()
  data_result["minimum"] = filtered_df["meter_reading"].min()

  return data_result

In [55]:
get_meter_reading_statistics(timestamp="2016-01-01")

{'range_days': 7,
 'maximum': 6355.91,
 'average': np.float64(168.93114850552308),
 'minimum': 0.01}

In [56]:
get_meter_reading_statistics("2016-01-01", 117, 5, True)

{'building_id': 117,
 'range_days': 5,
 'maximum': 32.612,
 'average': np.float64(3.60075),
 'minimum': 1.0}

In [57]:
get_meter_reading_statistics("2016-01-01", 117, 40, True)

{'building_id': 117,
 'range_days': 40,
 'maximum': 45.429,
 'average': np.float64(3.5567605633802812),
 'minimum': 1.0}

In [58]:
def rank_buildings_by_consumption(
  timestamp: str = Field(..., description="The reference timestamp for the search, in 'YYYY-MM-DD' format (without Z)."),
  range_days: int = Field(default=7, description="The number of days to include in the search range, used with 'before' and 'after' operators."),
  only_anomaly: bool = Field(default=False, description="Set to True to search only for data points marked as anomalies."),
  top_n: int = Field(default=5, description="The number of buildings to include in the ranking (e.g., 5 for the top 5).")
) -> dict:
  filtered_df = filter_data_by_timestamp(
      timestamp=timestamp,
      range_days=range_days,
      only_anomaly=only_anomaly
  )

  if filtered_df.empty:
      return {"message": "No data available for this period."}

  ranking = filtered_df.groupby('building_id')['meter_reading'].sum().sort_values(ascending=False).head(top_n)

  return ranking

In [59]:
rank_buildings_by_consumption("2016-01-01", 5, True, 3)

,meter_reading
building_id,
1258,61441.393
1251,16897.000
560,1974.100


In [60]:
data_agent = Agent(
  name="Data Agent",
  model=Gemini(id="gemini-2.0-flash"),
  instructions="""You are an assistant for question-answering tasks, please use your tools""",
  tools=[
    get_meter_reading_statistics,
    rank_buildings_by_consumption
  ],
  markdown=True
)

In [63]:
data_agent.print_response(
    "What was the statistics of meter reading from '2016-12-31' for building with id 1251?",
    stream=True
)

Output()

In [64]:
get_meter_reading_statistics(
    timestamp="2016-12-31",
    building_id=1251
)

{'building_id': 1251,
 'range_days': 7,
 'maximum': 428.0,
 'average': np.float64(369.875),
 'minimum': 320.0}

In [65]:
data_agent.print_response(
    "What was the highest meter reading, that is an anomaly, from '2016-12-31'?",
    stream=True
)

Output()

In [43]:
def get_current_date():
  return datetime.now()

In [45]:
report_agent = Agent(
  name="Report Agent",
  description="An Agent that can writes reports",
  model=Gemini(id="gemini-2.0-flash"),
  tools=[
      get_current_date
  ],
  instructions="""You are an assistant for that writes reports in a technical way. Start the report with:
    **Report for <CURRENT DATE>
  and finish with:
  ┃ Prepared by: Report Agent                                                                                 ┃                                                                                       ┃
  ┃ Locate: UNIBS""",
  markdown=True
)

In [46]:
supervisor_agent = Team(
    name="Reasoning Research Team",
    mode="coordinate",
    model=Gemini(id="gemini-2.0-flash"),
    members=[
        data_agent,
        report_agent
    ],
    tools=[ReasoningTools(add_instructions=True)],
    instructions=[
        "Collaborate to provide comprehensive report",
        "always use the data agent for data",
        "always use report agent to writes the report"
    ],
    markdown=True,
    show_members_responses=True,
    enable_agentic_context=True,
    add_datetime_to_instructions=True,
)

In [47]:
supervisor_agent.print_response(
    "Write a report for rank meter reading, that is an anomaly, from '2016-01-01'",
    stream=True
)

Output()